In [13]:
# ── Imports ───────────────────────────────────────────────────────────────
import os, re, pickle
import numpy as np
import faiss
from pathlib import Path
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from huggingface_hub import HfApi

from app2 import HF_TOKEN_AVANI

ROOT         = os.path.abspath(os.path.join(os.getcwd(), ".."))
DOCS_DIR     = os.path.join(ROOT, "dept_definition_docs")   # source docs folder
ARTIFACTS_DIR = os.path.join(ROOT, "rag_artifacts_v2") # output folder
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

load_dotenv(os.path.join(ROOT, "secrets.env"))
HF_TOKEN = os.getenv("HF_TOKEN_AVANI")
HF_REPO_ID = "avani1010/new_rag_index"

assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "HF_TOKEN missing"
print(f"✓ Root         : {ROOT}")
print(f"✓ Docs dir     : {DOCS_DIR}")
print(f"✓ Artifacts dir: {ARTIFACTS_DIR}")
print(f"✓ HF repo      : {HF_REPO_ID}")

✓ Root         : /Users/avani/IdeaProjects/customer-support-management
✓ Docs dir     : /Users/avani/IdeaProjects/customer-support-management/dept_definition_docs
✓ Artifacts dir: /Users/avani/IdeaProjects/customer-support-management/rag_artifacts_v2
✓ HF repo      : avani1010/new_rag_index


In [14]:
# ── Load the retrieval embedding model ────────────────────────────────────
RETRIEVAL_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(RETRIEVAL_MODEL)
DIM = embedder.get_sentence_embedding_dimension()

# Sanity check
test = embedder.encode("production server crashed", normalize_embeddings=True)
print(f"✓ Model        : {RETRIEVAL_MODEL}")
print(f"  Dim          : {DIM}   ← will be 384")
print(f"  Norm         : {np.linalg.norm(test):.4f}   ← should be 1.0")

✓ Model        : sentence-transformers/all-MiniLM-L6-v2
  Dim          : 384   ← will be 384
  Norm         : 1.0000   ← should be 1.0


In [15]:
# ── Parse dept definition docs and priority doc into chunks ───────────────

PRIORITY_FILE = "Priority_Escalation_Criteria.txt"

def split_into_sections(text: str, dept_label: str) -> list:
    """Split a doc on its ALL-CAPS section headers."""
    chunks = []
    sections = re.split(r"\n(?=[A-Z][A-Z\s&]{4,}\n[-=])", text)
    for sec in sections:
        sec = sec.strip()
        if len(sec) < 60:
            continue
        header_match = re.match(r"^([A-Z][A-Z\s&:]{2,})", sec)
        section_name = header_match.group(1).strip() if header_match else "OVERVIEW"
        chunks.append({
            "dept"      : dept_label,
            "section"   : section_name,
            "text"      : sec,
            "raw_body"  : sec,
            "chunk_type": "section",
        })
    return chunks


def sliding_window_chunks(text: str, dept_label: str, window=400, step=200) -> list:
    """Overlapping sliding window over raw text for dense coverage."""
    words  = text.split()
    chunks = []
    for i in range(0, max(1, len(words) - window + 1), step):
        span = " ".join(words[i : i + window])
        if len(span) < 60:
            continue
        chunks.append({
            "dept"      : dept_label,
            "section"   : f"SLIDING_WINDOW_{i}",
            "text"      : span,
            "raw_body"  : span,
            "chunk_type": "sliding",
        })
    return chunks


dept_chunks     = []
priority_chunks = []

for fname in sorted(os.listdir(DOCS_DIR)):
    if not fname.endswith(".txt"):
        continue
    fpath = os.path.join(DOCS_DIR, fname)
    text  = Path(fpath).read_text(encoding="utf-8")

    if fname == PRIORITY_FILE:
        # Priority doc — split by HIGH / MEDIUM / LOW sections
        for label in ["HIGH", "MEDIUM", "LOW"]:
            pattern = rf"PRIORITY:\s*{label}.*?(?=PRIORITY:\s*(?:HIGH|MEDIUM|LOW)|$)"
            m = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
            if m:
                body = m.group(0).strip()
                priority_chunks.append({
                    "dept"         : "PRIORITY",
                    "section"      : label,
                    "priority_hint": label,
                    "text"         : body,
                    "raw_body"     : body,
                    "chunk_type"   : "section",
                })
    else:
        # Dept definition doc — derive label from filename
        dept_label = fname.replace(".txt", "").replace("_", " ")
        sec_chunks = split_into_sections(text, dept_label)
        slw_chunks = sliding_window_chunks(text, dept_label)
        dept_chunks.extend(sec_chunks)
        dept_chunks.extend(slw_chunks)
        print(f"  {fname:<45}  sections:{len(sec_chunks)}  sliding:{len(slw_chunks)}")

print(f"\n✓ Dept definition chunks : {len(dept_chunks)}")
print(f"✓ Priority chunks        : {len(priority_chunks)}")
print(f"  Sections               : {[c['section'] for c in priority_chunks]}")

  Billing_and_Payments.txt                       sections:4  sliding:1
  Customer_Service.txt                           sections:4  sliding:1
  General_Inquiry.txt                            sections:4  sliding:1
  Human_Resources.txt                            sections:4  sliding:1
  Returns_and_Exchanges.txt                      sections:4  sliding:1
  Sales_and_Pre-Sales.txt                        sections:4  sliding:1
  Service_Outages_and_Maintenance.txt            sections:4  sliding:1
  Technical & IT Support.txt                     sections:4  sliding:1

✓ Dept definition chunks : 40
✓ Priority chunks        : 3
  Sections               : ['HIGH', 'MEDIUM', 'LOW']


In [16]:
# ── Embed dept definition chunks → HNSW FAISS index ───────────────────────
print(f"Embedding {len(dept_chunks)} dept definition chunks...")
dept_texts       = [c["text"] for c in dept_chunks]
dept_embeddings  = embedder.encode(
    dept_texts,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True,
).astype("float32")
print(f"✓ Dept embeddings shape: {dept_embeddings.shape}  ← should be (N, 384)")

# HNSW index — fast approximate search, good for larger doc sets
hnsw_index = faiss.IndexHNSWFlat(DIM, 32)
hnsw_index.hnsw.efConstruction = 200
hnsw_index.hnsw.efSearch        = 64
hnsw_index.add(dept_embeddings)
print(f"✓ HNSW dept index built — {hnsw_index.ntotal} vectors")

In [17]:
# ── Embed priority chunks → flat exact FAISS index ────────────────────────
# Only 3 chunks (HIGH/MEDIUM/LOW) so flat exact search is fine
print(f"Embedding {len(priority_chunks)} priority chunks...")
priority_texts      = [c["text"] for c in priority_chunks]
priority_embeddings = embedder.encode(
    priority_texts,
    normalize_embeddings=True,
    show_progress_bar=False,
).astype("float32")

priority_index = faiss.IndexFlatIP(DIM)
priority_index.add(priority_embeddings)
print(f"✓ Priority flat index built — {priority_index.ntotal} vectors")

Embedding 3 priority chunks...
✓ Priority flat index built — 3 vectors


In [18]:
# ── Build BM25 index over dept definition chunks ───────────────────────────
def tokenize_for_bm25(text: str) -> list:
    return re.sub(r"[^\w\s]", " ", text.lower()).split()

tokenized_corpus = [tokenize_for_bm25(c["text"]) for c in dept_chunks]
bm25             = BM25Okapi(tokenized_corpus)
print(f"✓ BM25 index built over {len(tokenized_corpus)} dept definition chunks")

✓ BM25 index built over 40 dept definition chunks


In [19]:
# ── Quick retrieval sanity check before saving ─────────────────────────────
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def test_retrieve(query, top_k=3):
    q_emb = embedder.encode(query, normalize_embeddings=True).astype("float32").reshape(1, -1)
    _, ids = hnsw_index.search(q_emb, top_k)
    candidates = [dept_chunks[i] for i in ids[0]]
    ce_scores  = cross_encoder.predict([[query, c["text"]] for c in candidates])
    ranked = sorted(zip(candidates, ce_scores), key=lambda x: x[1], reverse=True)
    print(f"Query: {query!r}")
    for c, s in ranked:
        print(f"  dept={c['dept']:<35} section={c['section']:<20} CE={s:.3f}")
    print()

test_retrieve("I was charged twice for my subscription")
test_retrieve("My internet connection keeps dropping")
test_retrieve("I want to return a defective product")
test_retrieve("URGENT: our entire system is down, losing money every minute")

Query: 'I was charged twice for my subscription'
  dept=Billing and Payments                section=SLIDING_WINDOW_0     CE=-5.168
  dept=Billing and Payments                section=OVERVIEW             CE=-6.388
  dept=Billing and Payments                section=ROUTING DECISION GUIDANCE CE=-6.958

Query: 'My internet connection keeps dropping'
  dept=Technical & IT Support              section=REPRESENTATIVE EXAMPLES CE=-10.227
  dept=Service Outages and Maintenance     section=REPRESENTATIVE EXAMPLES CE=-10.685
  dept=Technical & IT Support              section=ROUTING DECISION GUIDANCE CE=-11.319

Query: 'I want to return a defective product'
  dept=Returns and Exchanges               section=OVERVIEW             CE=2.388
  dept=Returns and Exchanges               section=ROUTING DECISION GUIDANCE CE=-2.491
  dept=Billing and Payments                section=REPRESENTATIVE EXAMPLES CE=-10.898

Query: 'URGENT: our entire system is down, losing money every minute'
  dept=Service Outag

In [20]:
# ── Save all artifacts locally ─────────────────────────────────────────────

# 1. Dept definition FAISS index  (renamed from rag_compliance_index)
dept_faiss_path = os.path.join(ARTIFACTS_DIR, "rag_dept_index.faiss")
faiss.write_index(hnsw_index, dept_faiss_path)
print(f"✓ Dept FAISS saved      → {dept_faiss_path}")

# 2. BM25 index
bm25_path = os.path.join(ARTIFACTS_DIR, "rag_bm25_index.pkl")
with open(bm25_path, "wb") as f:
    pickle.dump({"bm25": bm25, "tokenized_corpus": tokenized_corpus}, f)
print(f"✓ BM25 saved            → {bm25_path}")

# 3. Dept chunk metadata  (renamed from rag_compliance_metadata)
dept_meta_path = os.path.join(ARTIFACTS_DIR, "rag_dept_metadata.pkl")
with open(dept_meta_path, "wb") as f:
    pickle.dump(dept_chunks, f)
print(f"✓ Dept metadata saved   → {dept_meta_path}")

# 4. Priority FAISS index
priority_faiss_path = os.path.join(ARTIFACTS_DIR, "rag_priority_index.faiss")
faiss.write_index(priority_index, priority_faiss_path)
print(f"✓ Priority FAISS saved  → {priority_faiss_path}")

# 5. Priority chunk metadata
priority_meta_path = os.path.join(ARTIFACTS_DIR, "rag_priority_metadata.pkl")
with open(priority_meta_path, "wb") as f:
    pickle.dump(priority_chunks, f)
print(f"✓ Priority metadata     → {priority_meta_path}")

print(f"\n✓ All 5 artifacts saved to {ARTIFACTS_DIR}")

✓ Dept FAISS saved      → /Users/avani/IdeaProjects/customer-support-management/rag_artifacts_v2/rag_dept_index.faiss
✓ BM25 saved            → /Users/avani/IdeaProjects/customer-support-management/rag_artifacts_v2/rag_bm25_index.pkl
✓ Dept metadata saved   → /Users/avani/IdeaProjects/customer-support-management/rag_artifacts_v2/rag_dept_metadata.pkl
✓ Priority FAISS saved  → /Users/avani/IdeaProjects/customer-support-management/rag_artifacts_v2/rag_priority_index.faiss
✓ Priority metadata     → /Users/avani/IdeaProjects/customer-support-management/rag_artifacts_v2/rag_priority_metadata.pkl

✓ All 5 artifacts saved to /Users/avani/IdeaProjects/customer-support-management/rag_artifacts_v2


In [22]:
# ── Upload all artifacts to HuggingFace ───────────────────────────────────
api = HfApi(token=HF_TOKEN_AVANI)

artifacts = [
    (dept_faiss_path,       "rag_dept_index.faiss"),
    (bm25_path,             "rag_bm25_index.pkl"),
    (dept_meta_path,        "rag_dept_metadata.pkl"),
    (priority_faiss_path,   "rag_priority_index.faiss"),
    (priority_meta_path,    "rag_priority_metadata.pkl"),
]

for local_path, hf_filename in artifacts:
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=hf_filename,
        repo_id=HF_REPO_ID,
        repo_type="model",
    )
    print(f"  ✓ Uploaded → {hf_filename}")

print(f"\n✓ All artifacts live at https://huggingface.co/{HF_REPO_ID}")
print("  stage2b_retriever.py will now load the correct 384-dim indexes.")

  ✓ Uploaded → rag_priority_metadata.pkl

✓ All artifacts live at https://huggingface.co/avani1010/new_rag_index
  stage2b_retriever.py will now load the correct 384-dim indexes.
